In [1]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

In [2]:
def build_vedral_circuit(a, b):
    qc = QuantumCircuit(13, 5)

    # q1,q4,q7,q10 = bits of A (LSB to MSB)
    # q2,q5,q8,q11 = bits of B (LSB to MSB)

    # encode A
    if a & 1: qc.x(1)
    if a & 2: qc.x(4)
    if a & 4: qc.x(7)
    if a & 8: qc.x(10)

    # encode B
    if b & 1: qc.x(2)
    if b & 2: qc.x(5)
    if b & 4: qc.x(8)
    if b & 8: qc.x(11)

    qc.barrier()

    # adder gates
    qc.ccx(1,2,3)
    qc.ccx(4,5,6)
    qc.ccx(7,8,9)
    qc.ccx(10,11,12)
    
    qc.cx(1,2)
    qc.cx(4,5)
    qc.cx(7,8)
    qc.cx(10,11)
    
    qc.ccx(0,2,3)
    qc.ccx(3,5,6)
    qc.ccx(6,8,9)
    qc.ccx(9,11,12)
    
    qc.cx(10,11)
    qc.cx(10,11)
    qc.cx(9,11)
    
    qc.ccx(6,8,9)
    qc.cx(7,8)
    qc.ccx(7,8,9)
    qc.cx(7,8)
    qc.cx(6,8)
    
    qc.ccx(3,5,6)
    qc.cx(4,5)
    qc.ccx(4,5,6)
    qc.cx(4,5)
    qc.cx(3,5)
    
    qc.ccx(0,2,3)
    qc.cx(1,2)
    qc.ccx(1,2,3)
    qc.cx(1,2)
    qc.cx(0,2)
    
    # =====================================================
    # MEASUREMENTS
    # =====================================================
    
    
    qc.measure(2,0);
    qc.measure(5,1);
    qc.measure(8,2);
    qc.measure(11,3);
    qc.measure(12,4);

    return qc

In [5]:
import numpy as np

shots = 100

# ====================================================
# DEPOLARIZING NOISE MODEL
# ====================================================

p = 0.0378

error_1 = depolarizing_error(p, 1)
error_2 = depolarizing_error(p, 2)
error_3 = depolarizing_error(p, 3)

noise_model = NoiseModel()

noise_model.add_all_qubit_quantum_error(error_1, ['x'])
noise_model.add_all_qubit_quantum_error(error_2, ['cx'])
noise_model.add_all_qubit_quantum_error(error_3, ['ccx'])

sim = AerSimulator(noise_model=noise_model)

# ====================================================
# NUMPY ARRAYS FOR METRICS
# ====================================================

ER_arr = np.zeros((16, 16))
NMED_arr = np.zeros((16, 16))
MRED_arr = np.zeros((16, 16))

for a in range(16):
    for b in range(16):

        qc = build_vedral_circuit(a, b)

        compiled = transpile(qc, sim)

        counts = sim.run(
            compiled,
            shots=shots
        ).result().get_counts()

        correct = a + b
        correct_output = format(correct, '05b')

        correct_counts = counts.get(correct_output, 0)

        ER = 1 - correct_counts / shots

        total_ED = 0
        total_relative_ED = 0

        for output, freq in counts.items():

            noisy_decimal = int(output, 2)

            ED = abs(correct - noisy_decimal)

            total_ED += ED * freq

            if correct != 0:
                total_relative_ED += (ED / correct) * freq

        mean_ED = total_ED / shots

        D = 30

        NMED = mean_ED / D
        MRED = total_relative_ED / shots

        ER_arr[a, b] = ER
        NMED_arr[a, b] = NMED
        MRED_arr[a, b] = MRED

# ====================================================
# OVERALL METRICS
# ====================================================

er = np.mean(ER_arr)
nmed = np.mean(NMED_arr)
mred = np.mean(MRED_arr)

print("Depolarizing Noise\n")

print(f"ER (%) : {er * 100:.4f}")
print(f"NMED   : {nmed:.6f}")
print(f"MRED   : {mred:.6f}")

# Flatten to 1D if needed
er_array = ER_arr.flatten()

print("\nER Array:")
# print(er_array)

Depolarizing Noise

ER (%) : 48.7773
NMED   : 0.074879
MRED   : 0.215674

ER Array:
